# CyTOF Correction

This notebook applies the full correction pipeline to a pre-loaded cell matrix.

**Prerequisites:** Run `DataPreparation.ipynb` first to generate the annotated matrix CSV and study config.

**Correction steps applied:**
1. **Time drift correction** — cell-type-specific intra-run normalization using a stable marker (e.g. DNA)
2. **Arcsinh transformation** — cofactor = 5
3. **Batch correction** — inter-plate quantile alignment using an anchor sample

**One run per panel:** For large datasets (e.g. D24 with 15 plates), `DataPreparation` saves one
study config per panel. Run this notebook **once per panel**, pointing `study_config_path` at the
relevant config each time:
```
study_config_cohort1_D24_Panel1.json  →  one notebook run
study_config_cohort1_D24_Panel2.json  →  second notebook run
```
For small datasets where both panels share a single study config (e.g. D0), one run handles both.

In [ ]:
import sys
import os
import json
import pandas as pd

# Add the project root to the Python path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src import apply_correction, get_config, CyTOFContext
# Also import load_data_from_omiq if you want to skip DataPreparation.ipynb
from src import load_data_from_omiq


## Load study config and cell matrix

Point `study_config_path` at the `study_config.json` saved by `DataPreparation.ipynb`.  
Everything else — `ctx`, `dataset`, `panel`, `anchor_name`, matrix paths — is derived from it automatically.

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────────
study_config_path = "../results/study_config.json"
# ─────────────────────────────────────────────────────────────────────────────

with open(study_config_path) as f:
    study_config = json.load(f)

config = get_config(config_dict=study_config["cytof_config"])
ctx    = CyTOFContext(config)

dataset     = study_config["dataset"]
panel       = study_config["panel"]
anchor_name = ctx.anchors.get(dataset)

with open(study_config["outputs"]["markers_path"]) as f:
    markers = json.load(f)

matrix_path = study_config["outputs"]["matrix_path"]
header = pd.read_csv(matrix_path, nrows=0).columns
category_cols = {
    "original_filename", "dataset", "panel", "plate",
    "sample_name", "condition", "celltypes", "group",
}
float_cols = [c for c in header if c not in category_cols]
dtype_map = {c: "float32" for c in float_cols}
dtype_map.update({c: "category" for c in header if c in category_cols})
matrix = pd.read_csv(matrix_path, dtype=dtype_map)

print(f"Dataset    : {dataset}")
print(f"Panel      : {panel}")
print(f"Anchor     : {anchor_name}")
print(f"ref_plate  : {ctx.ref_plate}")
print(f"Matrix     : {matrix.shape}")
print(f"Markers ({len(markers)}): {markers[:5]}...")
display(matrix.head(3))

In [ ]:
print("Columns:", list(matrix.columns))
print("\nCells per plate:")
# display(matrix.groupby(['plate', 'sample_name']).size().rename('n_cells').reset_index())

print(f"\nAnchor '{anchor_name}' present in plates:")
print(sorted(matrix[matrix['sample_name'] == anchor_name]['plate'].unique()))
print(
    matrix.groupby(['panel','plate','sample_name'])
    .size()
    .to_string()
)

## Alternative: load inline (skip DataPreparation.ipynb)

If you want to load and correct in a single notebook, replace the CSV load above  
with `load_data_from_omiq`. This cell is kept for reference — **do not run it**  
if you already loaded `matrix` from CSV above.

In [ ]:
# # --- Option B: load directly from raw files ---
# # (Replace paths and values to match your cohort)

# import pandas as _pd

# _omiq_meta = _pd.read_excel("../data/cohort_2/OMIQ_metadata-OHSU 18 patient dataset.xlsx", sheet_name=2)

# _metadata = _pd.DataFrame()
# _metadata['file_name']   = _omiq_meta['Filename'].str.replace('.fcs', '.csv', regex=False)
# _metadata['plate']       = _metadata['file_name'].apply(lambda x: x.split('_')[0] if 'Plate' in x else 'Plate1')
# _metadata['sample_name'] = _metadata.apply(lambda r: r['file_name'].replace(r['plate']+'_','').replace('.csv',''), axis=1)
# _metadata['dataset']     = "OHSU_18"
# _metadata['panel']       = "Panel1"
# _metadata['condition']   = _omiq_meta['Ex-vivo treatment'].values

# _config = get_config(config_dict={
#     "data_dir": "../data/cohort_2/omiq_exported_data/",
#     "output_dir": "../results/Correction/",
#     "ref_plate": "Plate1",
#     "anchors": {"STUDY_2": "ANCHOR_SAMPLE"},
#     "meta_cols": ["original_filename","Event_length","Time","dataset","panel","plate","sample_name","condition"],
#     "celltypes": ["89Y_CD45+/T-cells","89Y_CD45+/B-cells","immature-like","megakaryocytic","myeloid-enriched"],
#     "celltype2correct": ["89Y_CD45+/T-cells","89Y_CD45+/B-cells","immature-like","megakaryocytic","myeloid-enriched"],
#     "stable_marker": "DNA",
#     "true_markers": ["CD45","DNA","CD3","CD4","CD8","CD19","CD14","CD33","CD56","CD34"],
#     "metal_to_marker": {},
#     "celltype_gating": ["89Y_CD45+/T-cells","89Y_CD45+/B-cells","immature-like","megakaryocytic","myeloid-enriched"]
# })
# _ctx = CyTOFContext(_config)

# matrix, markers = load_data_from_omiq(
#     data_dir    = _ctx.data_dir,
#     metadata_df = _metadata,
#     ctx         = _ctx,
#     dataset     = "OHSU_18",
# )
# anchor_name = "ANCHOR_SAMPLE"

## Apply correction

In [ ]:
import gc

# apply_correction returns TWO matrices:
#   matrix_intermediate : time-corrected + arcsinh, BEFORE batch correction
#                         -> "After" in time-correction visualization
#                         -> diagnostic baseline between time and batch steps
#   matrix_corrected    : time + arcsinh + batch corrected (full pipeline)
#
# Both visualization notebooks load raw from matrix_path and apply arcsinh inline,
# so neither needs a separate "raw arcsinh" file:
#   Visualization_TimeCorrection  : raw arcsinh (inline) vs matrix_intermediate
#   Visualization_BatchCorrection : raw arcsinh (inline) vs matrix_corrected

matrix_intermediate, matrix_corrected = apply_correction(
    df          = matrix,
    markers     = markers,
    anchor_name = anchor_name,
    ctx         = ctx,
)
del matrix
gc.collect()

print(f"\nIntermediate (time-corrected + arcsinh) shape : {matrix_intermediate.shape}")
print(f"Corrected (full pipeline) shape               : {matrix_corrected.shape}")
display(matrix_corrected.head())


In [ ]:
# BATCH CORRECTION DIAGNOSTIC
# Tells you whether the correction actually did anything, and how much.
import matplotlib.pyplot as plt
import numpy as np

numeric_markers = [m for m in markers if m in matrix_intermediate.columns]

# 1. Per-plate mean absolute change
print("=== Mean absolute change per plate (intermediate -> corrected) ===")
per_plate = {}
for plate in sorted(matrix_intermediate["plate"].unique()):
    before = matrix_intermediate[matrix_intermediate["plate"] == plate][numeric_markers]
    after  = matrix_corrected[matrix_corrected["plate"] == plate][numeric_markers]
    mad    = (after.values - before.values).mean(axis=0)  # signed mean shift
    per_plate[plate] = mad
    print(f"  {plate}: mean_abs_change={np.abs(mad).mean():.4f}  max_change={np.abs(mad).max():.4f}")

# 2. Anchor distributions across plates (before correction) for the most variable marker
anchor_name_diag = ctx.anchors.get(dataset)
df_anchor_inter = matrix_intermediate[matrix_intermediate["sample_name"] == anchor_name_diag]

marker_variance = df_anchor_inter.groupby("plate")[numeric_markers].median().std()
top_marker = marker_variance.idxmax()
print(f"\nMost variable marker across plates (anchor, pre-correction): {top_marker}  (std={marker_variance[top_marker]:.4f})")

plates = sorted(df_anchor_inter["plate"].unique())
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, (df_src, title) in zip(axes, [
    (matrix_intermediate, "Before batch correction"),
    (matrix_corrected,    "After  batch correction"),
]):
    df_a = df_src[df_src["sample_name"] == anchor_name_diag]
    for plate in plates:
        vals = df_a[df_a["plate"] == plate][top_marker].dropna()
        ax.hist(vals, bins=60, alpha=0.5, density=True, label=plate)
    ax.set_title(f"{title}\n{top_marker} - anchor only")
    ax.set_xlabel(top_marker)
    ax.legend(fontsize=7)
plt.suptitle(f"Anchor '{anchor_name_diag}' distribution across plates", fontsize=13)
plt.tight_layout()
plt.show()


## Inspect the corrected matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

# Quick sanity check: compare batch-correction effect in arcsinh space.
# Both "before" and "after" are arcsinh-transformed so the scales are comparable.
# "Before batch" = matrix_intermediate (time-corrected + arcsinh, no batch correction)
# "After batch"  = matrix_corrected    (time-corrected + arcsinh + batch-corrected)
marker_to_plot = markers[0]   # change to any marker of interest
plates = sorted(matrix_intermediate['plate'].unique())

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (df_plot, title) in zip(axes, [
    (matrix_intermediate, 'Before batch correction (arcsinh)'),
    (matrix_corrected,    'After batch correction (arcsinh)'),
]):
    for plate in plates:
        vals = df_plot[df_plot['plate'] == plate][marker_to_plot].dropna()
        ax.hist(vals, bins=80, alpha=0.5, density=True, label=plate)
    ax.set_title(title)
    ax.set_xlabel(marker_to_plot)
    ax.legend(fontsize=8)

plt.suptitle(f"Batch correction effect: {marker_to_plot}", fontsize=14)
plt.tight_layout()
plt.show()


## Save the corrected matrix

In [ ]:
out_dir = os.path.dirname(study_config["outputs"]["matrix_corrected_path"])
os.makedirs(out_dir, exist_ok=True)

corrected_path = study_config["outputs"]["matrix_corrected_path"]
corrected_name = os.path.basename(corrected_path)

# Derive intermediate path from study_config if pre-populated,
# otherwise derive from the corrected path basename.
if study_config["outputs"].get("matrix_intermediate_path"):
    intermediate_path = study_config["outputs"]["matrix_intermediate_path"]
else:
    intermediate_path = os.path.join(out_dir, corrected_name.replace("_corrected.csv", "_intermediate.csv"))

# Save the intermediate matrix (time-corrected + arcsinh, before batch correction).
# Loaded by Visualization_TimeCorrection.ipynb as "After time correction".
matrix_intermediate.to_csv(intermediate_path, index=False)
print(f"Intermediate (time-corrected + arcsinh) saved to: {intermediate_path}")

# Save the fully corrected matrix (time + arcsinh + batch).
# Loaded by Visualization_BatchCorrection.ipynb as "After" (raw vs full pipeline).
matrix_corrected.to_csv(corrected_path, index=False)
print(f"Corrected (full pipeline) saved to:               {corrected_path}")

# Patch study_config with the resolved intermediate path so visualization notebooks can find it.
study_config["outputs"]["matrix_intermediate_path"] = intermediate_path
with open(study_config_path, "w") as f:
    json.dump(study_config, f, indent=2)
print(f"study_config updated:                             {study_config_path}")
